# Lab 2 — Vaizdų klasifikavimas

Dominykas Misius, 2213772

**Modelis:** ResNet18 (fine-tuned)

**Duomenų rinkinys:** Open Images V6

**Klasės:** Cat, Dog, Bird

## 1. Aplinkos paruošimas

In [ ]:
!pip install -q fiftyone

from google.colab import drive
import os
import json

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/GMM_Lab2'
LABELS_PATH = os.path.join(BASE_DIR, 'labels.json')
MODEL_PATH = os.path.join(BASE_DIR, 'best_model.pth')

os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base dir: {BASE_DIR}")

## 2. Duomenų atsisiuntimas iš OpenImages V6

Atsisiunčiame vaizdus su klasifikavimo etiketėmis naudojant FiftyOne.
Kiekvienam vaizdui priskiriame vieną dominuojančią klasę.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

CHOSEN_CLASSES = ["Cat", "Dog", "Bird"]
NUM_CLASSES = len(CHOSEN_CLASSES)

# --- Train split ---
print("Loading TRAIN split...")
train_dataset = foz.load_zoo_dataset(
    "open-images-v6",
    split="train",
    max_samples=1500,
    classes=CHOSEN_CLASSES,
    label_types=["classifications"],
)

# --- Validation split (used as test) ---
print("Loading TEST (validation) split...")
test_dataset = foz.load_zoo_dataset(
    "open-images-v6",
    split="validation",
    max_samples=500,
    classes=CHOSEN_CLASSES,
    label_types=["classifications"],
)

TRAIN_DIR = os.path.dirname(next(iter(train_dataset)).filepath)
TEST_DIR = os.path.dirname(next(iter(test_dataset)).filepath)
print(f"Train images: {TRAIN_DIR}")
print(f"Test images:  {TEST_DIR}")

In [ ]:
def extract_single_labels(dataset, chosen_classes):
    """Assign each image exactly one class label (first matching)."""
    labels = {}
    for sample in dataset:
        filename = os.path.basename(sample.filepath)
        pos_labels = set()
        if sample.positive_labels is not None:
            for cls in sample.positive_labels.classifications:
                if cls.label in chosen_classes:
                    pos_labels.add(cls.label)
        if len(pos_labels) == 1:
            labels[filename] = list(pos_labels)[0]
    return labels


if os.path.exists(LABELS_PATH):
    print("Labels file found, loading from disk.")
    with open(LABELS_PATH, 'r') as f:
        all_labels = json.load(f)
    train_labels = all_labels['train']
    test_labels = all_labels['test']
else:
    train_labels = extract_single_labels(train_dataset, CHOSEN_CLASSES)
    test_labels = extract_single_labels(test_dataset, CHOSEN_CLASSES)

    all_labels = {'train': train_labels, 'test': test_labels}
    with open(LABELS_PATH, 'w') as f:
        json.dump(all_labels, f, indent=2)
    print(f"Labels saved to: {LABELS_PATH}")

print(f"\nTrain images: {len(train_labels)}")
for cls in CHOSEN_CLASSES:
    print(f"  {cls}: {sum(1 for v in train_labels.values() if v == cls)}")

print(f"\nTest images: {len(test_labels)}")
for cls in CHOSEN_CLASSES:
    print(f"  {cls}: {sum(1 for v in test_labels.values() if v == cls)}")

## 3. Dataset ir DataLoader

Mokymo duomenims taikomi augmentacijos (random crop, flip), testavimui — tik resize + center crop.
Naudojamas standartinis ImageNet normalizavimas.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

CLASS_TO_IDX = {cls: i for i, cls in enumerate(CHOSEN_CLASSES)}
IDX_TO_CLASS = {i: cls for cls, i in CLASS_TO_IDX.items()}

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class ImageClassificationDataset(Dataset):
    def __init__(self, data_dir, labels_dict, transform):
        self.transform = transform
        self.samples = []
        for filename, label in labels_dict.items():
            filepath = os.path.join(data_dir, filename)
            if os.path.exists(filepath):
                self.samples.append((filepath, CLASS_TO_IDX[label]))
        print(f"Dataset: {len(self.samples)} images from {data_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            image = self.transform(Image.open(path).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)
        return image, label


train_ds = ImageClassificationDataset(TRAIN_DIR, train_labels, train_transform)
test_ds  = ImageClassificationDataset(TEST_DIR, test_labels, test_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 4. Modelio apibrėžimas

Naudojame iš anksto apmokytą ResNet18, pakeičiame paskutinį sluoksnį (`fc`) į 3 klases.
Užšaldome visus sluoksnius, išskyrus `layer4` ir `fc` — tai leidžia greitai ir efektyviai fine-tune'inti.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 + fc
for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} parameters ({100*trainable/total:.1f}%)")

## 5. Modelio mokymas

In [ ]:
NUM_EPOCHS = 10
LR = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

best_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Eval ---
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    print(f"  loss={train_loss:.4f}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}  lr={scheduler.get_last_lr()[0]:.1e}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  -> Saved best model (val_acc={best_acc:.4f})")

    scheduler.step()

print(f"\nBest validation accuracy: {best_acc:.4f}")

## 6. Modelio įkėlimas ir predikcijos ant testavimo aibės

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

all_preds, all_true = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Test inference"):
        images = images.to(DEVICE)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(labels.numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_true)
print(f"Test samples: {len(y_true)}")

## 7. Metrikos

- **Klasifikavimo matrica** (Confusion Matrix)
- **Tikslumas** (Accuracy)
- **Precizija** (Precision)
- **Atkūrimas** (Recall)
- **F1**

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
)
import matplotlib.pyplot as plt

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(cm, display_labels=CHOSEN_CLASSES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Klasifikavimo matrica')
plt.tight_layout()
plt.show()

# --- Metrics ---
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

print(f"Tikslumas (Accuracy):   {acc:.4f}")
print(f"Precizija (Precision):  {prec:.4f}")
print(f"Atkūrimas (Recall):     {rec:.4f}")
print(f"F1:                     {f1:.4f}")

print("\n--- Detalios metrikos kiekvienai klasei ---")
print(classification_report(y_true, y_pred, target_names=CHOSEN_CLASSES, zero_division=0))

## 8. Testavimas su pasirinktais vaizdais

In [ ]:
import io
from google.colab import files
from IPython.display import Image as IPImage, display

print("Įkelkite vaizdą (-us):")
uploaded_files = files.upload()

if not uploaded_files:
    print("Jokių vaizdų neįkelta.")
else:
    for filename, file_bytes in uploaded_files.items():
        print(f"\n--- {filename} ---")
        display(IPImage(data=file_bytes, width=300))

        img_tensor = test_transform(
            Image.open(io.BytesIO(file_bytes)).convert("RGB")
        ).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(img_tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        pred_idx = probs.argmax()
        print(f"  Predikcija: {IDX_TO_CLASS[pred_idx]}  ({probs[pred_idx]:.2%})")
        print("  Tikimybės:")
        for i, cls in enumerate(CHOSEN_CLASSES):
            bar = '█' * int(probs[i] * 30)
            print(f"    {cls:<6s} {probs[i]:.4f} {bar}")